In [1]:
from src.improved_model import  BinarizingCNN
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch

import numpy as np




cnn_model = BinarizingCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet_v2.pth'))
cnn_model.to(device)
cnn_model.eval_mode()


# type(cnn_model.layer1.weight)

#
# def convert_model_to_binarized(model: BinarizingNetwork):
#
#     ...

C:\Users\frrit\AppData\Local\Temp\ipykernel_10196\2008466882.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path /

In [2]:
print(cnn_model.layer2.bias)
cnn_model.binarize_weights()
print(cnn_model.layer2.bias)


Parameter containing:
tensor([3.5579, 3.7151, 3.5012, 1.6117, 1.8762, 1.1311, 1.2187, 1.0080, 1.2166,
        1.2233, 3.1612, 3.5776, 3.6541, 1.1799, 2.8980, 3.3143, 2.9908, 1.0243,
        4.0283, 3.7748, 3.2350, 1.3466, 2.9761, 1.2125, 1.4341, 1.2316, 3.5730,
        3.5492, 3.3030, 1.4650, 1.1516, 3.4523, 1.1210, 2.3967, 3.1849, 1.0245,
        2.6062, 1.0312, 1.0464, 1.2643, 1.7882, 1.4444, 1.1849, 2.2475, 3.7073,
        1.1186, 1.0435, 3.9899, 2.4857, 1.0574, 2.8670, 3.4544, 1.1704, 1.3532,
        1.0055, 1.2132, 1.1886, 5.3536, 1.0785, 3.7698, 3.6585, 1.5462, 1.3456,
        1.1088, 1.3134, 1.8635, 1.1846, 1.2837, 4.3617, 3.2203, 3.5909, 3.2441,
        1.2966, 1.2681, 3.3181, 1.2108, 1.8885], device='cuda:0',
       requires_grad=True)
Parameter containing:
tensor([3., 3., 3., 1., 1., 1., 1., 1., 1., 1., 3., 3., 3., 1., 2., 3., 2., 1.,
        4., 3., 3., 1., 2., 1., 1., 1., 3., 3., 3., 1., 1., 3., 1., 2., 3., 1.,
        2., 1., 1., 1., 1., 1., 1., 2., 3., 1., 1., 3., 2., 1.,

Parameter containing:
tensor([[[[-1.1695e+01,  7.8597e+00,  5.3642e+00,  4.9883e+00,  4.5989e+00],
          [-1.1288e+01, -1.7723e+00,  4.1320e+00,  5.5355e+00,  4.7975e+00],
          [-6.4392e+00, -2.8317e+01, -1.8932e+01, -7.1104e+00,  4.4085e+00],
          [ 9.0338e+00, -2.3755e+00, -1.5543e+01, -1.7282e+01, -3.6223e+00],
          [ 8.1471e+00,  2.9603e+00,  6.1901e+00,  8.5447e-01, -8.7988e+00]]],


        [[[-1.0800e+00,  6.9136e-01,  1.4560e+00,  8.1455e+00,  1.0874e+01],
          [ 4.1405e-01, -7.5549e+00,  1.5906e+01,  6.6820e+00, -5.5459e+00],
          [-6.8675e+00,  1.1305e+01,  2.4623e+01, -1.0206e+01, -4.0771e+00],
          [-1.2927e+01,  2.6781e+01, -8.1445e+00, -5.9198e+00,  3.3597e+00],
          [ 9.7673e+00,  2.1235e+01, -2.2610e+00,  1.3801e+00, -4.2793e+00]]],


        [[[-1.7534e+01, -1.4527e+01, -5.6378e+00, -1.0051e+01, -5.3308e-02],
          [-1.1649e+01,  3.6585e+00, -2.4641e+00, -6.2849e+00,  7.2246e+00],
          [-1.8545e+01, -2.5524e+00, -3.0856e+

In [3]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=int(1e3), shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=5000, shuffle=False)

In [5]:
train_data, _  = next(iter(train_dataloader))
train_data = train_data.to(device)
cnn_model(train_data)


RuntimeError: "addmm_cuda" not implemented for 'Long'

In [4]:
cnn_model = BinarizingCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet_v2.pth'))
cnn_model.to(device)
cnn_model.eval_mode()

old_bias = cnn_model.layer3.bias.clone()
old_output = cnn_model(train_data)

cnn_model.third_layer = lambda x: torch.where(cnn_model.layer3(x) >= 0, torch.tensor(1), torch.tensor(-1))
new_bias = torch.floor(old_bias.clone())
cnn_model.layer3.bias = torch.nn.Parameter(new_bias)
new_output = cnn_model(train_data)

assert  torch.all(old_output == new_output)

C:\Users\frrit\AppData\Local\Temp\ipykernel_35996\606248072.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path / '

RuntimeError: "addmm_cuda" not implemented for 'Long'

In [ ]:
new_output.dtype

In [ ]:
new_output[old_output != new_output]


In [ ]:
# Check if all values in each column are the same
same_values_per_column = (x == x[0]).all(dim=0)

# Get the indices of columns where all values are the same
identical_columns = torch.nonzero(same_values_per_column, as_tuple=True)[0]

print(identical_columns)


In [ ]:
print(torch.floor(old_bias))
old_bias